# 02 — Features (Elo & match features)

> **This notebook mirrors `src/worldcup/features/elo.py` and `src/worldcup/features/build.py`.**
> The code is the source of truth; this is a readable walkthrough that reproduces the **same** results. Nothing here is re-implemented — we just call the package.

In [1]:
from worldcup.data.load import load_results
from worldcup.features.elo import build_current_ratings, match_weight
from worldcup.features.build import build_features, FEATURE_COLUMNS

matches = load_results()
print(f"{len(matches):,} matches")

49,449 matches


## Tournament-weighted Elo

Ratings update after every match; K is scaled by match importance (`match_weight`).

In [2]:
for t in ["Friendly", "FIFA World Cup qualification", "UEFA Euro", "FIFA World Cup"]:
    print(f"{t:32} weight = {match_weight(t)}")

elo = build_current_ratings(save=False)
elo.top(10)

Friendly                         weight = 0.5
FIFA World Cup qualification     weight = 1.2
UEFA Euro                        weight = 1.5
FIFA World Cup                   weight = 2.0


,team,elo
0,Argentina,2251.4
1,Spain,2229.8
2,France,2189.9
3,England,2165.5
4,Colombia,2108.8
5,Brazil,2084.7
6,Netherlands,2070.4
7,Norway,2067.2
8,Germany,2062.4
9,Morocco,2048.2


## Leak-free match features

`build_features` does a strict chronological pass: read state → features, **then** update. So no feature ever sees the match it describes.

In [3]:
feats = build_features(matches)
print("features:", FEATURE_COLUMNS)
print("shape:", feats.shape)
print(feats["result"].value_counts(normalize=True).round(3).to_string())
feats[FEATURE_COLUMNS + ["result"]].tail()

features: ['elo_diff', 'form_points_diff', 'form_gd_diff', 'neutral']
shape: (49147, 10)
result
H    0.490
A    0.282
D    0.228


,elo_diff,form_points_diff,form_gd_diff,neutral,result
49142,263.627105,-0.2,-0.4,1,D
49143,284.916969,0.4,1.2,1,H
49144,456.045287,1.0,1.8,1,H
49145,139.479019,0.2,0.6,1,H
49146,-116.320359,-1.6,-3.2,1,A
